In [ ]:
using LinearAlgebra
"""
The function naive takes in 2 vectors with length n of k-tuples, 
these represent the input and output data (points before and after a rigid transformation) 
for calculating the translation vector b and a rotation matrix Q. It returns Q and b as a 2-tuple
"""
function naive(X, Y)
    if isempty(X) || isempty(Y)
        throw(ArgumentError("X and Y must not be empty"))
    elseif length(X) != length(Y)
        throw(DimensionMismatch("lengths of the input vectors X and Y arent equal"))
    elseif length(X[1]) != length(Y[1])
        throw(DimensionMismatch("sizes of tuples arent equal")) 
    end

    n = length(X); #num of points
    k = length(X[1]); #the dimension 
    
    if n < 2*k
        throw(ArgumentError("our assumption is that n >= 2k"))
    end

    X1 = [stack(X)' ones(n)]
    Y1 = stack(Y)'
    
    
    Qb = X1\Y1
    b = Qb[end, :]
    Q = Qb[1:end-1, :]'
    F = qr(Q)

    #After learning a bit about what qr returns and potentially distorts, it would seem the
    #sign of the columns in Q from qr can differ compared to the ones in the original Q
    #and that happens because we get the original matrix as a product of columns of Q -> q_i with diagonal 
    #elements of R->r_ii, so in order to keep the same orientation we "force" R_ii to be positive, 
    #although we dont explicitly do that to R because it would be a waste, so we just act like it is and
    #fix our orthogonal Q accordingly 
    d = sign.(diag(Matrix(F.R))) #here we collect signs of diagonal elements of R into a vector  
    Q_new = Matrix(F.Q) * Diagonal(d) #we have to multiply Q_new with a diagonal matrix not a vector
    return (Q_new, b)
end

naive

In [7]:
y1 = [(1,2), (3, 4), (5, 6)]
y11 = stack(y1)'
ones(3)
y = [y11 ones(3)]
F = qr(y)
q = Matrix(F.Q)
r = Matrix(F.R)
d = sign.(diag(r))
q *= Diagonal(d)

3×3 Matrix{Float64}:
 0.169031   0.897085   0.408248
 0.507093   0.276026  -0.816497
 0.845154  -0.345033   0.408248

In [10]:
y_test = [(5, 6), (7, 8), (10, 11), (12, 12)]
x_test = [(1, 2), (3, 4), (8, 9), (13, 14)]
naive(x_test, y_test)

([-0.5904102148425776 0.8071033256092689; -0.8071033256092689 -0.5904102148425775], [3.098943323727182, 3.9490874159462033])